# Sentiment Timeline Analysis

This notebook presents the sentiment timeline analysis


In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import sys

In [ ]:
root_dir = Path().resolve().parents[0]
data_dir = root_dir / "data" / "rmp_ucsd_reviews.csv"
sys.path.append(str(root_dir))
df = pd.read_csv(data_dir)


In [ ]:
COLUMNS_TO_KEEP = [
    "date",
    "class",
    "qualityRating",
    "difficultyRatingRounded",
    "grade",
    "comment",
]
df = df[COLUMNS_TO_KEEP].copy()
valid_course_pattern = r"^[A-Z]{3,}[0-9]+[A-Z]?$"
df = df[df["class"].str.match(valid_course_pattern, na=False)].copy()
df

In [ ]:
df = df.dropna().copy()
df = df.rename(columns={
    "qualityRating": "quality",
    "clarityRatingRounded": "clarity",
    "difficultyRatingRounded": "difficulty",
})
df

In [ ]:
from src.sentiment import normalize_dates, filter_courses, plot_course_ratings

course = "PHYS2A"

df_clean = normalize_dates(df=df)
df_filtered = filter_courses(df=df_clean,
                             course=course,
                             start_date=pd.Timestamp("2016-01-01"),
                             end_date=pd.Timestamp("2026-12-31"))
fig, ax = plot_course_ratings(df=df_filtered, course=course)
plt.show()

In [ ]:
from src.sentiment import smooth_ratings, plot_smoothed_ratings

df_smoothed = smooth_ratings(df_filtered)

fig, ax = plot_smoothed_ratings(df_smoothed, course)
plt.show()

In [ ]:
from src.sentiment import load_releases, overlay_releases

releases_df = load_releases(root_dir / "data" / "chatgpt_model_updates.csv")

fig, ax = plot_smoothed_ratings(df_smoothed, course)
overlay_releases(ax, releases_df)

plt.show()

In [ ]:
course_list = ["MATH20C", "PHYS2A", "CHEM6A", "ECE35", "ECE45", "HUM3", "MMW11"]

for course in course_list:
    df_filtered = filter_courses(df=df_clean,
                             course=course,
                             start_date=pd.Timestamp("2016-01-01"),
                             end_date=pd.Timestamp("2026-12-31"))
    df_smoothed = smooth_ratings(df_filtered)
    fig, ax = plot_smoothed_ratings(df_smoothed, course)
    overlay_releases(ax, releases_df)

plt.show()
